# M2S-Bench Protocol-B Pipeline Demo

This notebook runs the adapter-ready Protocol B loop: load fixed demo cells, apply the visibility manifest, run a template solver, evaluate the standardized JSONL submission, and print one leaderboard row.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'demo_cells.jsonl').exists():
    subprocess.run([sys.executable, 'scripts/materialize_pipeline_demo.py', '--n-cells', '50'], cwd=ROOT, check=True)
ROOT

In [ ]:
sys.path.insert(0, str(ROOT))
from m2sbench.evaluator import load_jsonl
from m2sbench.visibility import load_visibility_manifest, method_visible_cell, assert_no_hidden_fields

cells = load_jsonl(ROOT / 'data' / 'demo_cells.jsonl')
manifest = load_visibility_manifest(ROOT / 'data' / 'visibility_manifest.json')
visible_cells = [method_visible_cell(cell, manifest) for cell in cells]
for cell in visible_cells:
    assert_no_hidden_fields(cell)
len(visible_cells), visible_cells[0].keys()

In [ ]:
subprocess.run([sys.executable, 'scripts/run_demo.py', '--solver', 'solvers/template_solver.py'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/validate_submission.py', 'outputs/demo_submission.jsonl', '--challenge', 'data/demo_cells.jsonl'], cwd=ROOT, check=True)
Path('outputs/demo_submission.jsonl').read_text(encoding='utf-8').splitlines()[0][:240]

In [ ]:
subprocess.run([sys.executable, 'scripts/evaluate_submission.py', 'outputs/demo_submission.jsonl'], cwd=ROOT, check=True)
json.loads((ROOT / 'outputs' / 'demo_scores.json').read_text(encoding='utf-8'))

In [ ]:
print((ROOT / 'outputs' / 'demo_leaderboard.md').read_text(encoding='utf-8'))